<a href="https://colab.research.google.com/github/tommypolpo/geron-hands_on_ML/blob/main/c5_ex7_ex8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV


X_moons, y_moons = make_moons(n_samples=10000, noise=0.4, random_state=42)

X_moons.shape

(10000, 2)

In [5]:
X_moons[:10]

array([[ 0.9402914 ,  0.12230559],
       [ 0.12454026, -0.42477546],
       [ 0.26198823,  0.50841438],
       [-0.49523824,  0.07258876],
       [-0.87941281,  0.54937303],
       [-0.24780176, -0.49459289],
       [ 1.3341025 ,  0.13489508],
       [-0.67741039,  0.0124898 ],
       [ 0.23045935, -0.18889569],
       [ 0.93385863,  0.02671338]])

Each point in the datasets is a point in the plane. So it has at most 2 features.

In [4]:
y_moons[:5]

array([1, 0, 0, 0, 0])

In [6]:
X_moons_train, X_moons_test, y_moons_train, y_moons_test = train_test_split(X_moons, y_moons, test_size=0.2, random_state=42)

### use cross-validation GridSearchCV to find good hyperparameters values for a
### DecisionTreeClassifier

parameter_grid = {
    'max_depth': [2, 3,4, 5, 6, 7, 8, 9, 10],
    'min_samples_split': [2, 3, 4, 5, 6, 7, 8, 9, 10], #how many samples a node can have before splitting
    'min_samples_leaf': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10] #maximum number of leaf nodes
}

grid_search = GridSearchCV(DecisionTreeClassifier(), parameter_grid, cv=5, scoring='accuracy')
grid_search.fit(X_moons_train, y_moons_train)





GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'max_depth': [2, 3, 4, 5, 6, 7, 8, 9, 10],
                         'min_samples_leaf': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
                         'min_samples_split': [2, 3, 4, 5, 6, 7, 8, 9, 10]},
             scoring='accuracy')

In [7]:
grid_search.best_params_

{'max_depth': 7, 'min_samples_leaf': 9, 'min_samples_split': 2}

In [10]:
print("Best CV Accuracy on the training set:", grid_search.best_score_)

Best CV Accuracy on the training set: 0.85625


In [9]:
predictions = grid_search.predict(X_moons_test)

accuracy = accuracy_score(y_moons_test, predictions)

print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.8665


In [13]:
from sklearn.model_selection import ShuffleSplit

# Initialize ShuffleSplit for 1000 splits with 100 samples each
# rs for random split
rs = ShuffleSplit(n_splits=1000, train_size=100, random_state=42)

# Store the subsets in lists
mini_sets = []

for train_index, _ in rs.split(X_moons_train):
    X_mini = X_moons_train[train_index]
    y_mini = y_moons_train[train_index]
    mini_sets.append((X_mini, y_mini))

# Verification: check the total number of subsets created
print(f"Total subsets generated: {len(mini_sets)}")
print(f"Shape of the first X subset: {mini_sets[0][0].shape}")

Total subsets generated: 1000
Shape of the first X subset: (100, 2)


In [15]:
from sklearn.base import clone
import numpy as np


# Get the best-parameter model from the grid search
best_tree_clf = grid_search.best_estimator_

# Create and train 1,000 separate decision trees
trained_forest = []
accuracy_scores = []
accuracy_scores_test = []


for X_mini, y_mini in mini_sets:
    # clone() creates a fresh, untrained model with the exact same hyperparameters
    mini_tree = clone(best_tree_clf)
    mini_tree.fit(X_mini, y_mini)
    trained_forest.append(mini_tree)
    accuracy = mini_tree.score(X_mini, y_mini)
    accuracy_scores.append(accuracy)
    y_predict = mini_tree.predict(X_moons_test)
    accuracy_test = accuracy_score(y_moons_test, y_predict)
    accuracy_scores_test.append(accuracy_test)



print(f"Successfully trained {len(trained_forest)} decision trees")
print(f"Mean accuracy score on the training sets: {np.mean(accuracy_scores)}")
print(f"Mean accuracy score on the test sets: {np.mean(accuracy_scores_test)}")


Successfully trained 1000 decision trees
Mean accuracy score on the training sets: 0.8618399999999999
Mean accuracy score on the test sets: 0.7999359999999999


For each test set instance, let's look at the 1000 predictions generated by the 1000 "small trees" and take the most frequent one.

In [16]:
from scipy.stats import mode

# Create an empty grid (matrix) to store all predictions.
# It has 1000 rows and 'len(X_test)' columns
# 'dtype=np.uint8' saves memory by storing the answers as small integers (0 or 1).
Y_pred = np.empty([1000, len(X_moons_test)], dtype=np.uint8)

# Loop through all 1,000 trained trees one by one.
# 'enumerate' gives us both the position index (0, 1, 2...) and the actual tree object.
for tree_index, tree in enumerate(trained_forest):
    # Ask the current tree to predict labels for the ENTIRE test set,
    # and save that row of predictions into our grid at the correct row index.
    Y_pred[tree_index] = tree.predict(X_moons_test)
    # e.g. Y_pred[17] is a array([0, 1, 1, 0, 0, 1, ..., 0, 1], dtype=uint8)
    # containing the predictions for the whole test set!

# Use a statistical 'mode' function to find the most frequent number.
# 'axis=0' tells it to look vertically down each column (looking at all 1,000 votes for a single test sample).
y_pred_majority_votes, n_votes = mode(Y_pred, axis=0)


In [18]:
accuracy_score(y_moons_test, y_pred_majority_votes.reshape([-1]))

0.8425